# 🦀 Day 4: SIMD
---

Today we explore **SIMD** — Single Instruction, Multiple Data.

- **What SIMD is**: Process 4–16 elements in parallel with one instruction
- **Why it matters**: 4x–16x throughput for data-parallel operations
- **Auto-vectorization**: The compiler already uses SIMD when it can
- **Helping the compiler**: Alignment, no branches, contiguous data
- **std::arch**: Platform-specific intrinsics (x86, ARM, etc.)
- **std::simd** (nightly): Portable SIMD abstractions
- **When SIMD helps**: Sum, dot product, distance; not when data is scattered

In [ ]:
// Sum of array — scalar version (compiler may auto-vectorize)
fn sum_scalar(arr: &[f32]) -> f32 {
    arr.iter().sum()
}

// Chunked processing pattern: process 4 elements at a time (SIMD-friendly logic)
fn sum_chunked(arr: &[f32]) -> f32 {
    let mut sum = 0.0f32;
    let chunks = arr.chunks_exact(4);
    let remainder = chunks.remainder();
    for chunk in chunks {
        // Simulated SIMD: add 4 elements (real SIMD would use _mm_add_ps etc.)
        sum += chunk[0] + chunk[1] + chunk[2] + chunk[3];
    }
    for &x in remainder {
        sum += x;
    }
    sum
}

let data: Vec<f32> = (0..1000).map(|i| i as f32).collect();
println!("Scalar sum: {}", sum_scalar(&data));
println!("Chunked sum: {}", sum_chunked(&data));

In [ ]:
// Dot product — classic SIMD target
fn dot_product(a: &[f32], b: &[f32]) -> f32 {
    assert_eq!(a.len(), b.len());
    a.iter().zip(b.iter()).map(|(x, y)| x * y).sum()
}

// Chunked dot product (SIMD-friendly structure)
fn dot_product_chunked(a: &[f32], b: &[f32]) -> f32 {
    assert_eq!(a.len(), b.len());
    let mut sum = 0.0f32;
    for (ca, cb) in a.chunks_exact(4).zip(b.chunks_exact(4)) {
        sum += ca[0]*cb[0] + ca[1]*cb[1] + ca[2]*cb[2] + ca[3]*cb[3];
    }
    let n = a.len() / 4 * 4;
    for i in n..a.len() {
        sum += a[i] * b[i];
    }
    sum
}

let a: Vec<f32> = (0..64).map(|i| i as f32).collect();
let b: Vec<f32> = (0..64).map(|i| (i * 2) as f32).collect();
println!("Dot product: {}", dot_product(&a, &b));
println!("Chunked dot: {}", dot_product_chunked(&a, &b));

## Helping the compiler auto-vectorize

- **Contiguous data**: Use slices, avoid linked structures
- **No branches in inner loop**: Use `select`-like logic instead of `if`
- **Alignment**: Aligned allocations can help (e.g. `align_to`)
- **std::arch** (x86): `_mm_add_ps`, `_mm_mul_ps` for manual SIMD
- **std::simd** (nightly): `Simd<f32, 4>` portable API

In [ ]:
// When SIMD helps vs when it doesn't
// GOOD: sum, dot product, distance, element-wise add/mul, min/max
// BAD: scattered memory access, heavy branching, small arrays

fn distance_squared(a: &[f32], b: &[f32]) -> f32 {
    a.iter().zip(b.iter()).map(|(x, y)| (x - y).powi(2)).sum()
}

let p: Vec<f32> = vec![1.0, 2.0, 3.0, 4.0];
let q: Vec<f32> = vec![0.0, 1.0, 2.0, 3.0];
println!("Distance²: {}", distance_squared(&p, &q));

## 📝 Exercise: Implement a chunked dot product

Implement `dot_product_chunked` that processes 8 elements at a time (instead of 4).
Handle the remainder. Compare with the scalar version using `Instant`.

In [ ]:
// YOUR CODE HERE
// fn dot_product_chunked_8(a: &[f32], b: &[f32]) -> f32 { ... }
// Benchmark: scalar vs chunked_4 vs chunked_8

## 🎯 Key Takeaways

1. **SIMD** = Single Instruction, Multiple Data — process many elements in parallel
2. **Auto-vectorization** often works; help with contiguous data and few branches
3. **Chunked processing** (4 or 8 elements) is a safe SIMD-friendly pattern
4. **std::arch** provides platform-specific intrinsics; **std::simd** (nightly) is portable
5. SIMD helps for sum, dot product, distance; less for scattered or branchy code
6. Measure before and after — sometimes scalar + auto-vectorization is enough

---
**Tomorrow:** Compile-time optimization